# AI Investment Analysis Agent

**DCF Valuation + Competitive Moat Analysis**  
Powered by LangGraph + GPT-4o mini + yfinance

---

## Workflow
1. **Cell 1** — Setup & configuration  
2. **Cell 2** — Run analysis (fetches data, computes DCF, pauses for review)  
3. **Cell 3** — Review and optionally override assumptions  
4. **Cell 4** — Resume analysis (sensitivity, moat, report)  
5. **Cell 5** — View results and charts  
6. **Cell 6** — Interactive Q&A  
7. **Cell 7** — Export reports

> ⚠️ **Disclaimer**: This tool is for educational purposes only. Not investment advice.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 1: Setup & Configuration
# ─────────────────────────────────────────────────────────────
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Load .env
from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, '.env'))

# Validate configuration
from config.settings import validate_keys, DEFAULT_LLM_MODEL, MARKET_RISK_PREMIUM
key_status = validate_keys()

print("=" * 60)
print("AI Investment Analysis Agent — Configuration")
print("=" * 60)
print(f"LLM Model:           {DEFAULT_LLM_MODEL}")
print(f"Market Risk Premium: {MARKET_RISK_PREMIUM:.1%}")
print()
print("API Keys:")
for service, configured in key_status.items():
    status = '✓ Configured' if configured else '✗ Not set'
    print(f"  {service:<20}: {status}")

if not key_status['openai']:
    print()
    print("⚠️  OpenAI API key not set. Add OPENAI_API_KEY to your .env file.")
    print("   LLM features will be unavailable but data fetching will still work.")

print()
print("Setup complete! Proceed to Cell 2 to start analysis.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2: Run Analysis
# ─────────────────────────────────────────────────────────────
# CONFIGURE: Enter ticker symbols to analyze
TICKERS = ["AAPL"]          # e.g. ["AAPL", "MSFT"] for multi-stock
THREAD_ID = "analysis_001"  # Unique ID for this analysis session

# Optional: Pre-supply assumption overrides (or leave empty)
# Format: {"TICKER": {"assumption_name": value}}
INITIAL_OVERRIDES = {}
# Example:
# INITIAL_OVERRIDES = {
#     "AAPL": {
#         "wacc": 0.09,
#         "terminal_growth_rate": 0.03,
#     }
# }

from src.agent.graph import run_analysis

# Run analysis — will pause before assumption review
graph, config = run_analysis(
    tickers=TICKERS,
    user_overrides=INITIAL_OVERRIDES,
    thread_id=THREAD_ID,
    with_interrupt=True,
)
# The graph pauses here after displaying assumptions above.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 3: Review & Override Assumptions
# ─────────────────────────────────────────────────────────────
# Review the assumptions displayed above.
# To override any value, fill in the dict below.
# Leave it empty ({}) to accept all agent suggestions.

# Available assumption keys:
#   wacc                   — Weighted average cost of capital (e.g. 0.09)
#   terminal_growth_rate   — Long-run FCF growth (e.g. 0.025)
#   revenue_growth_rates   — List of annual growth rates [yr1, yr2, ...]
#   operating_margin_target — Target EBIT margin (e.g. 0.28)
#   capex_percent_revenue  — CapEx as % of revenue (e.g. 0.04)
#   nwc_percent_revenue    — Net working capital as % of revenue (e.g. 0.02)
#   tax_rate               — Effective corporate tax rate (e.g. 0.21)

MY_OVERRIDES = {}  # <-- fill this in to override agent suggestions

# Example:
# MY_OVERRIDES = {
#     "AAPL": {
#         "wacc": 0.09,
#         "terminal_growth_rate": 0.025,
#         "revenue_growth_rates": [0.10, 0.08, 0.06, 0.05, 0.04],
#         "operating_margin_target": 0.30,
#     }
# }

if MY_OVERRIDES:
    for ticker, overrides in MY_OVERRIDES.items():
        print(f"Overrides for {ticker}:")
        for k, v in overrides.items():
            print(f"  {k}: {v}")
else:
    print("No overrides — accepting agent assumptions.")
print("\nProceed to Cell 4 to resume the analysis.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4: Resume Analysis
# ─────────────────────────────────────────────────────────────
from src.agent.graph import resume_after_review

final_state = resume_after_review(
    graph=graph,
    config=config,
    user_overrides=MY_OVERRIDES if MY_OVERRIDES else None,
)

print("\nAnalysis pipeline complete. Proceed to Cell 5 for results.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5: Results & Visualizations
# ─────────────────────────────────────────────────────────────
import pandas as pd

# ── Summary table ─────────────────────────────────────────────
print("=" * 70)
print("INVESTMENT ANALYSIS RESULTS")
print("=" * 70)

dcf_results = final_state.get('dcf_results', {})
dcf_assumptions = final_state.get('dcf_assumptions', {})
moat_analysis = final_state.get('moat_analysis', {})
sensitivity_results = final_state.get('sensitivity_results', {})
errors = final_state.get('errors', [])

summary_rows = []
for ticker in TICKERS:
    dr = dcf_results.get(ticker, {})
    da = dcf_assumptions.get(ticker, {})
    mo = moat_analysis.get(ticker, {})
    sr = sensitivity_results.get(ticker, {})
    mc = sr.get('monte_carlo_percentiles', {})
    summary_rows.append({
        'Ticker': ticker,
        'Current Price': f"${dr.get('current_price', 0):.2f}",
        'Intrinsic Value': f"${dr.get('intrinsic_value_per_share', 0):.2f}",
        'Upside/Downside': f"{dr.get('upside_downside_pct', 0):+.1%}",
        'Margin of Safety': f"{dr.get('margin_of_safety', 0):.1%}",
        'WACC': f"{da.get('wacc', 0):.2%}",
        'Terminal Growth': f"{da.get('terminal_growth_rate', 0):.2%}",
        'Moat Rating': mo.get('rating', 'N/A'),
        'Moat Score': f"{mo.get('total_score', 0):.0f}/50",
        'MC P50': f"${mc.get('p50', 0):.2f}",
    })

summary_df = pd.DataFrame(summary_rows).set_index('Ticker')
print(summary_df.to_string())

if errors:
    print(f"\n⚠️  Warnings ({len(errors)}):")
    for e in errors[-5:]:
        print(f"  • {e}")

In [ ]:
# ── Print full report ─────────────────────────────────────────
from IPython.display import Markdown, display

final_reports = final_state.get('final_reports', {})
for ticker in TICKERS:
    report = final_reports.get(ticker, f'No report generated for {ticker}')
    display(Markdown(report))
    print("\n" + "-" * 70 + "\n")

In [ ]:
# ── Charts ────────────────────────────────────────────────────
visualizations = final_state.get('visualizations', {})

TICKER = TICKERS[0]  # Change to view a different ticker
figs = visualizations.get(TICKER, {})

print(f"Available charts for {TICKER}: {list(figs.keys())}")
print("Run the cells below to display each chart.")

In [ ]:
# DCF Waterfall
if 'dcf_waterfall' in figs:
    figs['dcf_waterfall'].show()

In [ ]:
# Sensitivity Heatmap (WACC × Terminal Growth Rate)
if 'sensitivity_heatmap' in figs:
    figs['sensitivity_heatmap'].show()

# Also print the raw table
sens = sensitivity_results.get(TICKER, {})
if 'wacc_vs_tgr' in sens:
    print(f"\n{TICKER} Sensitivity Table ($):")
    print(sens['wacc_vs_tgr'].to_string())

In [ ]:
# Tornado Diagram
if 'tornado' in figs:
    figs['tornado'].show()

In [ ]:
# Moat Radar Chart
if 'moat_radar' in figs:
    figs['moat_radar'].show()

# Also print moat dimension scores
mo = moat_analysis.get(TICKER, {})
if mo:
    print(f"\n{TICKER} Moat Scores:")
    for dim in ['intangible_assets', 'switching_costs', 'network_effects', 'cost_advantages', 'efficient_scale']:
        print(f"  {dim.replace('_', ' ').title():25}: {mo.get(dim, 0):.1f}/10")
    print(f"  {'Total':25}: {mo.get('total_score', 0):.1f}/50  →  {mo.get('rating', 'N/A')}")

In [ ]:
# Monte Carlo Distribution
if 'monte_carlo' in figs:
    figs['monte_carlo'].show()

# Print percentile table
mc = sensitivity_results.get(TICKER, {}).get('monte_carlo_percentiles', {})
if mc:
    print(f"\n{TICKER} Monte Carlo Percentiles:")
    for pct, val in mc.items():
        print(f"  {pct:5}: ${val:.2f}")

In [ ]:
# Scenario Comparison
if 'scenarios' in figs:
    figs['scenarios'].show()

In [ ]:
# Historical Performance
if 'historical' in figs:
    figs['historical'].show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 6: Interactive Q&A
# ─────────────────────────────────────────────────────────────
from src.agent.graph import ask_question
from IPython.display import Markdown, display

# Ask a question about the analysis
QUESTION = "What are the key risks to the DCF valuation and how sensitive is it to the WACC assumption?"

print(f"Question: {QUESTION}")
print("-" * 60)

answer = ask_question(graph, config, QUESTION)
display(Markdown(answer))

In [ ]:
# Follow-up question (conversation is maintained across cells)
FOLLOW_UP = "If the terminal growth rate dropped to 2% from 2.5%, what would be the impact on intrinsic value?"

print(f"Question: {FOLLOW_UP}")
print("-" * 60)

answer2 = ask_question(graph, config, FOLLOW_UP)
display(Markdown(answer2))

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 7: Export Reports
# ─────────────────────────────────────────────────────────────
import os
import json
from pathlib import Path
from datetime import date

exports_dir = Path(project_root) / 'data' / 'exports'
exports_dir.mkdir(parents=True, exist_ok=True)

today = date.today().isoformat()

for ticker in TICKERS:
    # Markdown report
    report = final_reports.get(ticker, '')
    if report:
        md_path = exports_dir / f"{ticker}_{today}_report.md"
        md_path.write_text(report, encoding='utf-8')
        print(f"Saved: {md_path}")

    # JSON data export
    export_data = {
        'ticker': ticker,
        'analysis_date': today,
        'dcf_assumptions': dcf_assumptions.get(ticker, {}),
        'dcf_results': {
            k: v for k, v in dcf_results.get(ticker, {}).items()
            if not isinstance(v, list) or len(v) < 20  # skip large raw arrays
        },
        'moat_analysis': {
            k: v for k, v in moat_analysis.get(ticker, {}).items()
            if k != 'dimension_details'  # skip verbose nested dict
        },
        'sensitivity_scenarios': sensitivity_results.get(ticker, {}).get('scenarios', {}),
        'monte_carlo_percentiles': sensitivity_results.get(ticker, {}).get('monte_carlo_percentiles', {}),
    }

    json_path = exports_dir / f"{ticker}_{today}_data.json"
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, default=str)
    print(f"Saved: {json_path}")

    # Save charts as HTML (interactive)
    figs = visualizations.get(ticker, {})
    for chart_name, fig in figs.items():
        try:
            html_path = exports_dir / f"{ticker}_{today}_{chart_name}.html"
            fig.write_html(str(html_path))
            print(f"Saved chart: {html_path}")
        except Exception as e:
            print(f"Could not save {chart_name}: {e}")

print(f"\nAll exports saved to: {exports_dir}")